# Inference serving, modeled: roofline, batching, speculation

A trained LLM is only half the battle; the other half is *serving* it fast and cheap. This notebook makes the three levers that dominate serving economics concrete by **modeling** them from real hardware numbers — no GPU required, and we never pretend otherwise. Every hardware constant (HBM bandwidth, peak FLOP/s, weight bytes, KV bytes/token) is a named value from a real NVIDIA A100 serving Llama-3-8B, and every latency/throughput we print is *computed* from those constants, labelled as **modeled**.

We will build, in order:

1. **The decode roofline** — why decode is memory-bound, and why batching `B` requests raises throughput until the compute roofline.
2. **Static vs continuous batching** — why locked batches waste the GPU on ragged requests, and how in-flight batching fills the gaps.
3. **Speculative decoding** — the expected speedup as a function of the draft model's acceptance rate.

This is the same verified code that lives in the companion script `inference_serving.py` and is embedded in the concept page.


## The problem: a trained model that serves slowly is a model nobody can afford

Picture an 8B model on one A100. The weights are 16 GB in FP16. To generate **one** output token, the GPU must read **all 16 GB** of weights out of memory — and then do a tiny amount of math with them. At ~2 TB/s of memory bandwidth, that read alone is ~8 ms, so a single user gets ~125 tokens/s and the GPU's 312-TFLOP/s compute units sit ~99% idle, waiting on memory. That is the central pain of LLM serving: **decode is memory-bandwidth-bound**, and a GPU running one request is almost entirely wasted.

The cure is to make that expensive 16 GB weight read *count for more than one token*. Everything below is a different angle on that same idea.


> **Step 1 — pin the hardware and model constants.** These are the only inputs to every model below. They come from a real A100-80GB (FP16) serving Llama-3-8B with GQA-8. We compute the GPU's **roofline ridge point** — peak FLOP/s ÷ bandwidth ≈ 156 FLOP/byte — the arithmetic intensity above which a kernel is compute-bound and below which it is memory-bound.


In [1]:
import numpy as np

# Hardware: one real NVIDIA A100-80GB SXM, FP16.
HBM_BANDWIDTH_BYTES_PER_S = 2.0e12   # ~2.0 TB/s HBM2e bandwidth (the decode bottleneck)
PEAK_FLOPS = 312e12                   # ~312 TFLOP/s dense FP16 tensor-core throughput
GPU_FLOP_PER_BYTE = PEAK_FLOPS / HBM_BANDWIDTH_BYTES_PER_S   # ~156: the roofline ridge point

# Model: Llama-3-8B (GQA-8) -- the same example used in the KV-Cache chapter.
MODEL_PARAMS = 8.0e9
BYTES_PER_PARAM_FP16 = 2
WEIGHT_BYTES = MODEL_PARAMS * BYTES_PER_PARAM_FP16            # 16 GB streamed once per decode step
FLOPS_PER_PARAM_PER_TOKEN = 2                                 # one multiply-add per weight = 2 FLOPs
KV_BYTES_PER_TOKEN = 2 * 32 * 8 * 128 * BYTES_PER_PARAM_FP16  # 2(K,V)*layers*kv_heads*head_dim*bytes
MIB = 1024 * 1024

print("All values below are MODELED (no GPU, no wall-clock).")
print(f"NumPy: {np.__version__}")
print(f"HBM bandwidth: {HBM_BANDWIDTH_BYTES_PER_S/1e12:.1f} TB/s")
print(f"Peak compute : {PEAK_FLOPS/1e12:.0f} TFLOP/s")
print(f"Roofline ridge: {GPU_FLOP_PER_BYTE:.0f} FLOP/byte")
print(f"Weights      : {WEIGHT_BYTES/1e9:.0f} GB (FP16)")
print(f"KV per token : {KV_BYTES_PER_TOKEN/MIB:.3f} MiB (GQA-8)")


All values below are MODELED (no GPU, no wall-clock).
NumPy: 2.4.6
HBM bandwidth: 2.0 TB/s
Peak compute : 312 TFLOP/s
Roofline ridge: 156 FLOP/byte
Weights      : 16 GB (FP16)
KV per token : 0.125 MiB (GQA-8)


> **Step 2 — model one decode step.** A decode step must *both* stream bytes (every weight once, plus each sequence's KV cache) *and* do FLOPs (2 × params per token, for `batch` tokens). The GPU can't finish until the slower of the two is done, so the step time is the **max** of the memory time and the compute time. Notice what batching does: the 16 GB weight read is **fixed**, paid once no matter how many sequences ride along — so batching `B` requests reads the weights once but produces `B` tokens.


In [2]:
def decode_step_time_s(batch: int, context_tokens: int) -> tuple[float, float]:
    """Modeled (memory_time, compute_time) in seconds for one decode step at this batch."""
    bytes_moved = WEIGHT_BYTES + batch * context_tokens * KV_BYTES_PER_TOKEN  # weights once + KV per seq
    memory_time = bytes_moved / HBM_BANDWIDTH_BYTES_PER_S                      # bandwidth-bound time
    flops = FLOPS_PER_PARAM_PER_TOKEN * MODEL_PARAMS * batch                   # compute scales with batch
    compute_time = flops / PEAK_FLOPS                                          # compute-bound time
    return memory_time, compute_time

# Sanity-read: at batch 1, short context, memory time should dominate compute time by ~100x.
mem, cmp = decode_step_time_s(batch=1, context_tokens=256)
print(f"batch=1: memory {mem*1e3:.2f} ms  vs  compute {cmp*1e3:.3f} ms  ->  "
      f"{'memory-bound' if mem > cmp else 'compute-bound'} (ratio {mem/cmp:.0f}x)")


batch=1: memory 8.02 ms  vs  compute 0.051 ms  ->  memory-bound (ratio 156x)


> **Step 3 — sweep the batch size and watch throughput climb.** Now run the model across growing batch sizes at a short (256-token) context, where the weight read dominates. Throughput is `batch / step_time`: as `B` grows, each weight read is amortized across more tokens, so throughput rises almost linearly — until the compute term overtakes memory and the curve **saturates at the compute roofline**. The `bound` column flips from `memory` to `compute` exactly at that crossover.


In [3]:
BATCH_SIZES = (1, 8, 32, 64, 128, 156, 256, 512)
ROOFLINE_CONTEXT_TOKENS = 256  # short context: weights dominate, so the compute roofline is reachable

print(f"{'batch':>6} | {'latency/token':>14} | {'throughput':>16} | bound")
print("-" * 56)
rows = []
for batch in BATCH_SIZES:
    mem, cmp = decode_step_time_s(batch, ROOFLINE_CONTEXT_TOKENS)
    step_time = max(mem, cmp)                 # the slower resource sets the wall clock
    bound = "memory" if mem >= cmp else "compute"
    throughput = batch / step_time           # B tokens per step / step time
    rows.append((batch, step_time, throughput, bound))
    print(f"{batch:>6} | {step_time*1e3:>11.2f} ms | {throughput:>11.0f} tok/s | {bound}")

# The teaching claim, asserted: throughput rises with batch, then saturates; ends compute-bound.
assert rows[0][2] < rows[1][2] < rows[2][2], "throughput must rise with batch while memory-bound"
assert rows[-1][3] == "compute", "very large batch must reach the compute roofline"
print("\nasserts passed: throughput rises with batch, then saturates at the compute roofline.")


 batch |  latency/token |       throughput | bound
--------------------------------------------------------
     1 |        8.02 ms |         125 tok/s | memory
     8 |        8.13 ms |         983 tok/s | memory
    32 |        8.54 ms |        3748 tok/s | memory
    64 |        9.07 ms |        7053 tok/s | memory
   128 |       10.15 ms |       12614 tok/s | memory
   156 |       10.62 ms |       14693 tok/s | memory
   256 |       13.13 ms |       19500 tok/s | compute
   512 |       26.26 ms |       19500 tok/s | compute

asserts passed: throughput rises with batch, then saturates at the compute roofline.


> **Step 4 — find the crossover, and the long-context twist.** Solve `memory_time(B) == compute_time(B)` for the batch `B*` where decode turns compute-bound. At short context this crossover exists. But push context to 2048 tokens and the per-sequence KV term grows *faster* with `B` than the compute headroom — so decode **never** becomes compute-bound; throughput saturation comes from streaming the KV cache, not from compute. That is exactly why shrinking the KV cache (GQA, FP8) buys throughput at long context.


In [4]:
def roofline_crossover_batch(context_tokens: int) -> float | None:
    """Batch where compute overtakes memory, or None if it never does at this context."""
    compute_equiv_bytes_per_batch = (
        FLOPS_PER_PARAM_PER_TOKEN * MODEL_PARAMS / PEAK_FLOPS * HBM_BANDWIDTH_BYTES_PER_S
    )  # bytes of weight-read each extra unit of B can "hide" behind its compute
    kv_bytes_per_batch = context_tokens * KV_BYTES_PER_TOKEN
    net = compute_equiv_bytes_per_batch - kv_bytes_per_batch
    if net <= 0:
        return None  # KV growth outpaces compute headroom -> memory-bound forever
    return WEIGHT_BYTES / net

short = roofline_crossover_batch(256)
long = roofline_crossover_batch(2048)
print(f"crossover at  256-token context: B* = {short:.0f}")
print(f"crossover at 2048-token context: {'B* = %.0f' % long if long else 'never (memory-bound at all B)'}")


crossover at  256-token context: B* = 232
crossover at 2048-token context: never (memory-bound at all B)


## Lever 2: continuous batching — stop wasting slots on ragged requests

Batching raises throughput, but *how* you batch matters enormously. Real requests are **ragged**: one user wants 5 tokens, another wants 500. **Static** batching locks a batch until its *longest* request finishes — so the short requests' GPU slots sit idle, doing nothing, until the long one is done (head-of-line blocking). **Continuous** (in-flight) batching, introduced by Orca, refills a slot the instant its request finishes, from a queue of waiting work. Let's model both on the same ragged workload and the same engine width.


> **Step 5 — define the workload and the static-batching model.** Six ragged requests (4/6/20/60/5/8 output tokens) on a 3-slot engine. Static batching runs them in **waves** of 3; each wave is held for its longest request, so a wave with a 60-token request wastes the other two slots for ~55 steps. Utilization is useful token-steps ÷ offered slot-steps.


In [5]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Request:
    name: str
    output_tokens: int

def simulate_static_batching(requests, step_ms, slots):
    """Waves of `slots`; each wave locked until its longest request finishes (head-of-line blocking)."""
    total_steps = 0
    useful_steps = 0
    for start in range(0, len(requests), slots):           # carve requests into back-to-back waves
        wave = requests[start:start + slots]
        wave_length = max(r.output_tokens for r in wave)   # wave ends at its slowest request
        total_steps += wave_length                         # whole wave held this long, idle slots included
        useful_steps += sum(r.output_tokens for r in wave) # real token-steps in the wave
    makespan_ms = total_steps * step_ms
    utilization = useful_steps / (total_steps * slots)     # useful / offered slot-steps
    return makespan_ms, utilization

requests = [Request("a", 4), Request("b", 6), Request("c", 20),
            Request("d", 60), Request("e", 5), Request("f", 8)]
SLOTS, STEP_MS = 3, 10.0
static_makespan, static_util = simulate_static_batching(requests, STEP_MS, SLOTS)
print(f"static    : makespan {static_makespan:5.0f} ms | GPU util {static_util:.0%}")


static    : makespan   800 ms | GPU util 43%


> **Step 6 — the continuous-batching model, and the head-to-head.** Same engine, same requests — but now any slot that frees up is immediately backfilled from the queue. No wave barrier, no idle slots while work remains. We assert continuous **strictly beats** static on both makespan and utilization before printing, so the lesson is checked, not asserted by hope.


In [6]:
def simulate_continuous_batching(requests, step_ms, slots):
    """In-flight batching: a freed slot is backfilled from the queue the very next step (Orca)."""
    remaining = [r.output_tokens for r in requests]
    queue = list(range(len(requests)))
    active = []
    steps = 0
    busy_slot_steps = 0
    while queue or active:
        while len(active) < slots and queue:     # backfill every free slot before stepping
            active.append(queue.pop(0))
        busy_slot_steps += len(active)           # slots doing real work this step
        for i in active:
            remaining[i] -= 1                     # advance each active request one decode step
        active = [i for i in active if remaining[i] > 0]   # release finished slots now
        steps += 1
    makespan_ms = steps * step_ms
    utilization = busy_slot_steps / (steps * slots)
    return makespan_ms, utilization

cont_makespan, cont_util = simulate_continuous_batching(requests, STEP_MS, SLOTS)
assert cont_makespan < static_makespan, "continuous must finish strictly sooner here"
assert cont_util > static_util, "continuous must use the GPU strictly better here"
print(f"static     : makespan {static_makespan:5.0f} ms | GPU util {static_util:.0%}")
print(f"continuous : makespan {cont_makespan:5.0f} ms | GPU util {cont_util:.0%}")
print(f"-> {static_util:.0%} -> {cont_util:.0%} util, makespan {static_makespan/cont_makespan:.2f}x shorter")


static     : makespan   800 ms | GPU util 43%
continuous : makespan   640 ms | GPU util 54%
-> 43% -> 54% util, makespan 1.25x shorter


## Lever 3: speculative decoding — buy tokens with idle compute

Recall the GPU is ~99% idle during decode, starved for memory bandwidth. Speculative decoding spends that idle compute: a small, cheap **draft** model proposes `k` tokens, and the big **target** model **verifies all `k` in a single forward pass** (a mini-prefill that costs about one decode step), accepting the longest correct prefix. If the draft is good, you get several tokens for the price of one target pass.


> **Step 7 — model the expected speedup.** With per-token acceptance probability `alpha`, the expected number of tokens accepted per target pass is the truncated-geometric mean `E = (1 - alpha**(k+1)) / (1 - alpha)` (Leviathan et al., 2023). Each target pass also pays for `k` cheap draft steps at `draft_cost_ratio` of a target step. So `speedup = E / (1 + k * draft_cost_ratio)`. Sweep `alpha` and watch: high acceptance pays handsomely, but a low-acceptance draft *loses* (speedup < 1) because the overhead isn't earned back.


In [7]:
def speculative_speedup(alpha, draft_k, draft_cost_ratio):
    """Modeled wall-clock speedup of speculative decoding vs plain autoregressive decode."""
    if not 0.0 <= alpha < 1.0:
        raise ValueError("alpha must be in [0, 1)")
    # mean accepted draft prefix + 1 (the target's always-accepted bonus token after the last accepted draft)
    expected_accepted = (1.0 - alpha ** (draft_k + 1)) / (1.0 - alpha)
    target_passes_cost = 1.0 + draft_k * draft_cost_ratio              # one target pass + k draft steps
    return expected_accepted / target_passes_cost

print(f"{'acceptance alpha':>18} | {'expected speedup':>16}")
print("-" * 38)
for alpha in (0.1, 0.3, 0.5, 0.7, 0.8, 0.9):
    s = speculative_speedup(alpha=alpha, draft_k=4, draft_cost_ratio=0.1)
    print(f"{alpha:>18.1f} | {s:>14.2f}x")

# The two ends of the lesson, asserted.
assert speculative_speedup(0.8, 4, 0.1) > 1.0, "high acceptance should help"
assert speculative_speedup(0.1, 4, 0.4) < 1.0, "low acceptance + costly draft should hurt"
print("\nasserts passed: speedup climbs with acceptance; a poor draft can lose.")


  acceptance alpha | expected speedup
--------------------------------------
               0.1 |           0.79x
               0.3 |           1.02x
               0.5 |           1.38x
               0.7 |           1.98x
               0.8 |           2.40x
               0.9 |           2.93x

asserts passed: speedup climbs with acceptance; a poor draft can lose.


## Try it yourself

Before you change anything, **predict**, then run:

1. In **Step 3**, raise `ROOFLINE_CONTEXT_TOKENS` from 256 to 4096. Does the throughput at large batch go **up**, **down**, or **stay the same**? (Hint: more context → more KV bytes streamed per sequence → the memory term grows with batch, so the compute roofline moves further out and throughput at a given batch *drops*. Check the `bound` column — does it still reach `compute`?)
2. In **Step 6**, give the engine `SLOTS = 6` (as many slots as requests). Does continuous batching still beat static? (Hint: with a slot per request and no queue, there is nothing to backfill — the gap shrinks. The win *needs* more work than slots.)
3. In **Step 7**, set `draft_cost_ratio = 0.5` (an expensive draft). At what acceptance `alpha` does speculation stop paying off?


## What we saw

- **Decode is memory-bound** — at batch 1 the GPU spends ~8 ms reading 16 GB of weights to emit one token, and the compute units sit nearly idle.
- **Batching is the master throughput lever** — it amortizes the fixed weight read across `B` tokens, so throughput rose from ~125 to ~19,500 tok/s and then **saturated at the compute roofline** (ridge ≈ 156 FLOP/byte). At long context the KV term keeps decode memory-bound at every batch — which is *why* shrinking the cache buys speed.
- **Continuous batching** beat static on ragged requests by backfilling freed slots immediately — higher utilization and a shorter makespan — with no extra hardware.
- **Speculative decoding** spends idle compute to buy tokens: the speedup climbs with the draft's acceptance rate, and a cheap, well-aligned draft is what makes it pay.

Every number here was **modeled** from named A100/Llama-3-8B constants — reproducible on any machine, and grounded in the same arithmetic the production serving stack (vLLM, TGI, TensorRT-LLM) is built on. For the full story — PagedAttention, prefix caching, disaggregation, and the SLO/goodput picture — see the concept page.
